# 🧩 Mini-Lab: ReAct Agent

**Module 7: AI Agents** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** the ReAct (Reason + Act) framework and how it produces transparent reasoning traces before each action
2. **Compare** ReAct with Plan-and-Execute — two different strategies for how agents approach multi-step tasks
3. **Implement** a ReAct-style agent loop where the LLM explicitly reasons before selecting each tool call

## Target Concepts

| Concept | Description |
|---------|-------------|
| ReAct Framework | A prompting pattern where the LLM alternates between *Thought* (reasoning about what to do) and *Action* (executing a tool), producing transparent reasoning traces |
| Plan-and-Execute | An alternative strategy where the LLM creates a full plan upfront before executing any steps, contrasted with ReAct's step-by-step reasoning |

## Setup

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## From Agent Loop to ReAct

In `mini-agent-loop`, we built an agent that loops: **call LLM → execute tools → repeat**. It works, but the LLM's reasoning is invisible — we see *what* tool it called, but not *why*.

The **ReAct framework** (Reason + Act) fixes this by asking the LLM to produce an explicit **Thought** before each **Action**:

```
Thought: I need to convert miles to km first, then multiply.
Action:  convert_units(120, "miles", "km")
Observation: 193.12 km
Thought: Now I multiply 193.12 by 1.5.
Action:  calculate("multiply", 193.12, 1.5)
Observation: 289.68
Thought: I have the final answer.
Answer:  289.68 km
```

Each cycle is: **Thought → Action → Observation → repeat**.

This matters because:
- We can **debug** the agent's reasoning when things go wrong
- The LLM makes **better decisions** when forced to think explicitly
- Users can **trust** an agent whose reasoning is visible

## Step 1: Define Tools

We'll reuse the same calculator and unit converter from the previous lab — the tools themselves don't change with ReAct. What changes is **how we prompt** the LLM to use them.

In [2]:
def calculate(operation, a, b):
    """Perform a math operation on two numbers."""
    ops = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: division by zero"
    }
    result = ops.get(operation, "Error: unknown operation")
    return json.dumps({"operation": operation, "a": a, "b": b, "result": result})


def convert_units(value, from_unit, to_unit):
    """Convert between common units."""
    conversions = {
        ("miles", "km"): value * 1.60934,
        ("km", "miles"): value / 1.60934,
        ("fahrenheit", "celsius"): (value - 32) * 5 / 9,
        ("celsius", "fahrenheit"): value * 9 / 5 + 32,
        ("pounds", "kg"): value * 0.453592,
        ("kg", "pounds"): value / 0.453592,
    }
    key = (from_unit.lower(), to_unit.lower())
    result = conversions.get(key, "Error: unsupported conversion")
    if isinstance(result, float):
        result = round(result, 2)
    return json.dumps({"value": value, "from": from_unit, "to": to_unit, "result": result})


tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform a math operation (add, subtract, multiply, divide) on two numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["add", "subtract", "multiply", "divide"],
                        "description": "The math operation to perform."
                    },
                    "a": {"type": "number", "description": "The first number."},
                    "b": {"type": "number", "description": "The second number."}
                },
                "required": ["operation", "a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "convert_units",
            "description": "Convert a value from one unit to another. Supports: miles/km, fahrenheit/celsius, pounds/kg.",
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number", "description": "The numeric value to convert."},
                    "from_unit": {"type": "string", "description": "The source unit."},
                    "to_unit": {"type": "string", "description": "The target unit."}
                },
                "required": ["value", "from_unit", "to_unit"]
            }
        }
    }
]

available_functions = {
    "calculate": calculate,
    "convert_units": convert_units
}

print("Tools defined: calculate, convert_units")

Tools defined: calculate, convert_units


## Step 2: The ReAct System Prompt

The key difference between a plain agent loop and a ReAct agent is the **system prompt**. We instruct the LLM to produce a `Thought:` before every action. Since OpenAI's function calling happens in the API (not in text), we ask the LLM to put its reasoning in the `content` field of the assistant message, alongside the tool calls.

In [3]:
REACT_SYSTEM_PROMPT = """You are a helpful assistant that solves problems step by step.

IMPORTANT: Before EVERY action (tool call), you MUST first explain your reasoning
in your message content. Format your thinking as:

Thought: <your reasoning about what to do next and why>

Then make the appropriate tool call.

When you have the final answer and no more tools are needed, write:

Thought: <why you're done>
Answer: <your final answer>

Always think before acting. Never call a tool without explaining why first."""

print("ReAct system prompt defined.")

ReAct system prompt defined.


## Step 3: Build the ReAct Agent Loop

The loop structure is the same as `mini-agent-loop`, but now we **capture and display the reasoning trace** at each step. The LLM's `content` field contains the `Thought:` text, while its `tool_calls` field contains the action.

In [4]:
def run_react_agent(user_message, max_iterations=10):
    """Run a ReAct agent that thinks before each action."""
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    
    trace = []  # Collect the reasoning trace for display
    
    for i in range(1, max_iterations + 1):
        response = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=messages,
            tools=tools
        )
        
        assistant_msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason
        
        # Capture the Thought (reasoning in content)
        thought = assistant_msg.content or "(no explicit thought)"
        
        if finish_reason == "stop":
            # Final answer — no more tool calls
            trace.append(f"**Step {i} — Final:**\n{thought}")
            return thought, trace
        
        # Tool calls — log the Thought + Action + Observation
        messages.append(assistant_msg)
        
        step_log = f"**Step {i} — Thought:**\n{thought}\n"
        
        for tc in assistant_msg.tool_calls:
            args = json.loads(tc.function.arguments)
            fn = available_functions[tc.function.name]
            result = fn(**args)
            
            step_log += f"\n**Action:** `{tc.function.name}({args})`\n"
            step_log += f"**Observation:** `{result}`\n"
            
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            })
        
        trace.append(step_log)
    
    return "Max iterations reached.", trace

print("ReAct agent loop defined.")

ReAct agent loop defined.


## Step 4: See ReAct in Action

Let's ask a multi-step question and watch the full **Thought → Action → Observation** trace.

In [5]:
answer, trace = run_react_agent(
    "I ran a marathon (26.2 miles). Convert that to kilometers, "
    "then calculate how many minutes it would take at a pace of 5 minutes per km."
)

md("### 🔍 ReAct Reasoning Trace\n")
for step in trace:
    md(step)
    md("---")

### 🔍 ReAct Reasoning Trace


**Step 1 — Thought:**
Thought: First, I need to convert 26.2 miles to kilometers. Then, using the pace of 5 minutes per km, I will calculate the total time in minutes it would take to run that distance.

Let's start by converting miles to kilometers. I will use the conversion factor 1 mile ≈ 1.60934 km. I will perform this conversion first.

**Action:** `convert_units({'value': 26.2, 'from_unit': 'miles', 'to_unit': 'km'})`
**Observation:** `{"value": 26.2, "from": "miles", "to": "km", "result": 42.16}`


---

**Step 2 — Thought:**
Thought: The marathon distance in kilometers is approximately 42.16 km. Now, with a pace of 5 minutes per km, I will multiply 42.16 km by 5 minutes to find the total time in minutes.

Let's do that calculation now.

**Action:** `calculate({'operation': 'multiply', 'a': 42.16, 'b': 5})`
**Observation:** `{"operation": "multiply", "a": 42.16, "b": 5, "result": 210.79999999999998}`


---

**Step 3 — Final:**
Thought: The total time to run a marathon at a pace of 5 minutes per km is approximately 211 minutes. 

Answer: It would take about 211 minutes to run a marathon at that pace.

---

Notice how at each step, the LLM explicitly stated **why** it's taking that action. This is the core value of ReAct — the reasoning is visible and debuggable. Compare this to `mini-agent-loop` where we only saw *which* tool was called, not *why*.

## Step 5: ReAct vs. Plain Agent Loop

Let's run the same question through a **plain agent loop** (no ReAct prompt) and compare the visibility.

In [6]:
def run_plain_agent(user_message, max_iterations=10):
    """Run a plain agent loop (no ReAct prompting) for comparison."""
    messages = [{"role": "user", "content": user_message}]
    trace = []
    
    for i in range(1, max_iterations + 1):
        response = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=messages,
            tools=tools
        )
        
        assistant_msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason
        
        if finish_reason == "stop":
            trace.append(f"**Step {i} — Final:** {assistant_msg.content}")
            return assistant_msg.content, trace
        
        messages.append(assistant_msg)
        step_log = f"**Step {i}:**\n"
        step_log += f"Content: `{assistant_msg.content}`\n"
        
        for tc in assistant_msg.tool_calls:
            args = json.loads(tc.function.arguments)
            fn = available_functions[tc.function.name]
            result = fn(**args)
            step_log += f"Action: `{tc.function.name}({args})` → `{result}`\n"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        
        trace.append(step_log)
    
    return "Max iterations reached.", trace


question = "What is 250 pounds in kg? Then double that value."

md("### Plain Agent Loop")
plain_answer, plain_trace = run_plain_agent(question)
for step in plain_trace:
    md(step)

md("---\n### ReAct Agent Loop")
react_answer, react_trace = run_react_agent(question)
for step in react_trace:
    md(step)

### Plain Agent Loop

**Step 1:**
Content: `None`
Action: `convert_units({'value': 250, 'from_unit': 'pounds', 'to_unit': 'kg'})` → `{"value": 250, "from": "pounds", "to": "kg", "result": 113.4}`
Action: `convert_units({'value': 250, 'from_unit': 'pounds', 'to_unit': 'kg'})` → `{"value": 250, "from": "pounds", "to": "kg", "result": 113.4}`


**Step 2:**
Content: `None`
Action: `calculate({'operation': 'add', 'a': 113.4, 'b': 113.4})` → `{"operation": "add", "a": 113.4, "b": 113.4, "result": 226.8}`


**Step 3 — Final:** 250 pounds is approximately 113.4 kg. Doubling that value results in approximately 226.8 kg.

---
### ReAct Agent Loop

**Step 1 — Thought:**
Thought: To find out what 250 pounds is in kilograms, I need to convert from pounds to kilograms. After I get the converted value, I will then double it as requested. I will first perform the conversion, then multiply the result by 2. Let's start by converting 250 pounds to kilograms.

**Action:** `convert_units({'value': 250, 'from_unit': 'pounds', 'to_unit': 'kg'})`
**Observation:** `{"value": 250, "from": "pounds", "to": "kg", "result": 113.4}`


**Step 2 — Thought:**
(no explicit thought)

**Action:** `calculate({'operation': 'multiply', 'a': 113.4, 'b': 2})`
**Observation:** `{"operation": "multiply", "a": 113.4, "b": 2, "result": 226.8}`


**Step 3 — Final:**
Answer: 250 pounds is approximately 113.4 kg, and double that value is 226.8 kg.

The plain agent's `content` field is often `None` or empty — it goes straight to tool calls without explaining why. The ReAct agent always provides a `Thought:` explaining its reasoning.

Both arrive at the same answer, but the ReAct trace lets us **verify the reasoning**, not just the result.

## Step 6: Plan-and-Execute — An Alternative Strategy

ReAct reasons **one step at a time**: think → act → observe → think → act → ...

**Plan-and-Execute** takes a different approach: **plan everything upfront**, then execute the steps.

| Aspect | ReAct | Plan-and-Execute |
|--------|-------|------------------|
| **When planning happens** | Before each action (incremental) | All at once before any action |
| **Adapts to surprises** | Yes — each thought sees the latest observation | Less easily — plan is pre-set |
| **Good for** | Tasks where later steps depend on earlier results | Tasks where the full path is clear upfront |
| **Transparency** | Thought at each step | Full plan visible before execution |

Let's implement Plan-and-Execute to see the difference.

In [7]:
PLAN_EXECUTE_SYSTEM_PROMPT = """You are a helpful assistant that solves problems by planning first.

When given a task, FIRST create a numbered plan of ALL steps needed, like:

Plan:
1. <first step>
2. <second step>
3. <final step - provide the answer>

Then execute the plan step by step, making tool calls as needed.
After each tool result, note which step you just completed.

When all steps are done, provide the final answer."""

print("Plan-and-Execute system prompt defined.")

Plan-and-Execute system prompt defined.


In [8]:
def run_plan_execute_agent(user_message, max_iterations=10):
    """Run a Plan-and-Execute agent."""
    messages = [
        {"role": "system", "content": PLAN_EXECUTE_SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    
    trace = []
    
    for i in range(1, max_iterations + 1):
        response = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=messages,
            tools=tools
        )
        
        assistant_msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason
        content = assistant_msg.content or "(no content)"
        
        if finish_reason == "stop":
            trace.append(f"**Step {i} — Final:**\n{content}")
            return content, trace
        
        messages.append(assistant_msg)
        step_log = f"**Step {i}:**\n{content}\n"
        
        for tc in assistant_msg.tool_calls:
            args = json.loads(tc.function.arguments)
            fn = available_functions[tc.function.name]
            result = fn(**args)
            step_log += f"\n**Action:** `{tc.function.name}({args})`\n"
            step_log += f"**Observation:** `{result}`\n"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        
        trace.append(step_log)
    
    return "Max iterations reached.", trace

print("Plan-and-Execute agent defined.")

Plan-and-Execute agent defined.


In [9]:
question = (
    "I have 3 boxes, each weighing 50 pounds. "
    "What's the total weight in kilograms?"
)

md("### 📋 Plan-and-Execute Trace\n")
pe_answer, pe_trace = run_plan_execute_agent(question)
for step in pe_trace:
    md(step)
    md("---")

### 📋 Plan-and-Execute Trace


**Step 1:**
Plan:
1. Convert the weight of one box from pounds to kilograms.
2. Multiply the weight of one box in kilograms by 3 to get the total weight in kilograms.
3. Provide the total weight in kilograms as the final answer.

Let's start with step 1: convert 50 pounds to kilograms.

**Action:** `convert_units({'value': 50, 'from_unit': 'pounds', 'to_unit': 'kg'})`
**Observation:** `{"value": 50, "from": "pounds", "to": "kg", "result": 22.68}`


---

**Step 2:**
The weight of one box is approximately 22.68 kilograms. Now, I will multiply this by 3 to find the total weight.

**Action:** `calculate({'operation': 'multiply', 'a': 22.68, 'b': 3})`
**Observation:** `{"operation": "multiply", "a": 22.68, "b": 3, "result": 68.03999999999999}`


---

**Step 3 — Final:**
The total weight of the 3 boxes is approximately 68.04 kilograms.

---

Notice the difference: the Plan-and-Execute agent **laid out the full plan** (steps 1, 2, 3) in its first response, then executed them. The ReAct agent would have reasoned about each step only when it got there.

Let's compare both on the same question:

In [10]:
question = (
    "Convert 100 fahrenheit to celsius, "
    "then multiply the result by 3."
)

md("### 🧠 ReAct Trace\n")
react_answer, react_trace = run_react_agent(question)
for step in react_trace:
    md(step)

md("---\n### 📋 Plan-and-Execute Trace\n")
pe_answer, pe_trace = run_plan_execute_agent(question)
for step in pe_trace:
    md(step)

### 🧠 ReAct Trace


**Step 1 — Thought:**
Thought: First, I need to convert 100 Fahrenheit to Celsius. Then, I will multiply the Celsius result by 3. I will use the convert_units tool for the temperature conversion and then perform the multiplication.

Let's start with the conversion.

**Action:** `convert_units({'value': 100, 'from_unit': 'fahrenheit', 'to_unit': 'celsius'})`
**Observation:** `{"value": 100, "from": "fahrenheit", "to": "celsius", "result": 37.78}`


**Step 2 — Thought:**
(no explicit thought)

**Action:** `calculate({'operation': 'multiply', 'a': 37.78, 'b': 3})`
**Observation:** `{"operation": "multiply", "a": 37.78, "b": 3, "result": 113.34}`


**Step 3 — Final:**
Thought: The conversion from Fahrenheit to Celsius gives approximately 37.78°C. Multiplying this by 3 results in approximately 113.34. 

Answer: 113.34

---
### 📋 Plan-and-Execute Trace


**Step 1:**
Plan:
1. Convert 100 Fahrenheit to Celsius.
2. Multiply the Celsius temperature by 3.
3. Provide the final result.

Let's start with step 1: convert 100 Fahrenheit to Celsius.

**Action:** `convert_units({'value': 100, 'from_unit': 'fahrenheit', 'to_unit': 'celsius'})`
**Observation:** `{"value": 100, "from": "fahrenheit", "to": "celsius", "result": 37.78}`


**Step 2:**
(no content)

**Action:** `calculate({'operation': 'multiply', 'a': 37.78, 'b': 3})`
**Observation:** `{"operation": "multiply", "a": 37.78, "b": 3, "result": 113.34}`


**Step 3 — Final:**
The temperature of 100 Fahrenheit is approximately 37.78 Celsius. Multiplying this by 3 gives a result of approximately 113.34.

## When to Use Which?

| Scenario | Best Strategy | Why |
|----------|---------------|-----|
| Research tasks where each step depends on what you find | **ReAct** | Needs to adapt based on observations |
| Math/conversion pipelines where steps are clear | **Plan-and-Execute** | Full path known upfront |
| Debugging or auditing agent behavior | **ReAct** | Step-by-step reasoning is visible |
| Complex projects with many independent subtasks | **Plan-and-Execute** | Plan helps organize and track progress |

In practice, many agents **combine** both: plan first, then use ReAct-style reasoning during execution to adapt when things don't go as expected.

## 🎯 Summary

### Key Takeaways

1. **ReAct (Reason + Act)** — a prompting framework where the LLM produces an explicit Thought before each Action, creating a transparent and debuggable reasoning trace
2. **Plan-and-Execute** — an alternative strategy where the LLM generates a complete plan upfront before executing any steps, providing a clear roadmap for multi-step tasks
3. **ReAct adapts, Plan-and-Execute organizes** — ReAct is better when later steps depend on earlier observations; Plan-and-Execute is better when the full path is known in advance
4. **Both use the same agent loop** — the underlying loop (call LLM → execute tools → repeat) is identical; the difference is entirely in the system prompt that guides the LLM's reasoning style